In [1]:
# train.py
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any
from functools import partial

import numpy as np
# from datasets import load_dataset, Dataset, concatenate_datasets
from torch.utils.data import Dataset 
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer,
AutoModelForCausalLM
)
from transformers.trainer_utils import get_last_checkpoint
from accelerate import Accelerator
import torch
from sklearn.model_selection import train_test_split
# from tasks import render_example
import torch.nn as nn
import pandas as pd

import torch

from dataclasses import dataclass
from typing import Dict, Any, Callable, Optional
import evaluate
import shutil
import glob
from datetime import datetime
from peft import LoraConfig, get_peft_model

import os
import argparse
import numpy as np
import torch
import wandb
import evaluate
from datetime import datetime
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, TrainingArguments, Trainer
import sys
from pathlib import Path


In [2]:
odf = pd.read_csv("ldf_with_entropies_ntext.csv")

fdf = odf.copy()

# ldf["p_hat"] = pipe.predict_proba(ldf[cat_vars + cont_vars])[:, 1]
# def sentence_stats(group):
#     y_vals   = group[label_col].values      # actual 0/1 labels from raters
#     p_hats   = group["p_hat"].values        # model predicted probs per rater

#     p_1_obs = np.mean(y_vals)
#     p_0_obs = 1 - p_1_obs
#     H_obs = h_cat(p_1_obs, p_0_obs)
#     H_pred = h_cont(p_hats).mean()

#     return pd.Series({
#         "H_obs": H_obs,
#         "H_pred": H_pred,
#         "H_model": H_obs - H_pred,
#         "model_probs": str(p_hats.tolist())
#     })

# sent_level = (
#     ldf.groupby("sentence_id")
#        .apply(sentence_stats)
#        .reset_index()
# )

# ldf = ldf.merge(
#     sent_level,
#     on='sentence_id',
#     how='left'
# )
#originally we did the above. Model has issue with negative h_mod tho, so we changed to this for the final runs:

from sklearn.preprocessing import MinMaxScaler
fdf['H_obs_scaled'] = MinMaxScaler().fit_transform(fdf[['H_obs']])
fdf['H_pred_scaled'] = MinMaxScaler().fit_transform(fdf[['H_pred']])
fdf['H_model'] = (fdf['H_obs_scaled'] - fdf['H_pred_scaled']).clip(lower=0)

In [3]:
cat_cardinalities = {
    "words": 11267,
    "factual": 3,
    "group_id": 85,
    "type": 3,
    "topic": 14,
    "outlet": 8,
    "age": 54,
    "gender": 3,
    "education": 8,
    "native_english_speaker": 3,
    "political_ideology": 21,
    "followed_news_outlets": 551,
    "news_check_frequency": 6,
}

cat_embs_info = {}
for k, card in cat_cardinalities.items():
    if k not in fdf.columns:
        continue
    emb_dim = max(2, int(card ** 0.5) + 1) #sqrt(cardinality)) + 1
    emb_dim = min(50, emb_dim) #but not more than 50 - cough cough words
    cat_embs_info[k] = {"cardinality": card, "emb_dim": emb_dim}




class MTLDataset(Dataset):
    def __init__(self, 
                 df, 
                 cat_info, 
                 ambi_type, 
                 tokenizer,
                 zero_cats = False,
                 max_length=128):
        
        df = df.reset_index(drop=True).copy()
        self.texts = df["text"].tolist()
     
        for col in list(cat_info.keys()):
            df[col] = df[col].astype('category')
            try:
                df[col] = df[col].cat.codes
            except:
                print(col)

        cat_ids = df[list(cat_info.keys())].values   #[N, num_cat_vars]
        cat_ids = cat_ids.astype(int)

        if zero_cats:
            cat_ids = np.zeros_like(cat_ids)

    
        self.cat_ids = cat_ids
        self.y_bias = df["label_binary"].to_numpy(dtype=np.float32)
        self.y_ambi = df[ambi_type].to_numpy(dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        cats = self.cat_ids[idx]
        label_ambi = self.y_ambi[idx]
        label_bias = self.y_bias[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "x_cats": torch.tensor(cats, dtype=torch.long),
            "y_bias": torch.tensor(label_bias, dtype=torch.float),
            "y_ambi": torch.tensor(label_ambi, dtype=torch.float),
        }
        return item


In [4]:
class MTLClassifier(nn.Module):

    
    def __init__(self, 
                cats_info, 
                llm_pretrained_name,
                use_lora, 
                lora_r,
                ambi_loss_weight
                ):
        super().__init__()

        self.ambi_loss_weight = ambi_loss_weight
        self.llm = AutoModel.from_pretrained(
            llm_pretrained_name,
            output_hidden_states=True
        )

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=16,
            lora_dropout=0.05,
            bias="none",
            task_type="FEATURE_EXTRACTION",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        )

        if use_lora:
            self.llm = get_peft_model(self.llm, lora_config)
        d_model = self.llm.config.hidden_size

        #cats
        cat_cards = [cats_info[c]["cardinality"] for c in cats_info.keys()]
        cat_emb_dims = [cats_info[c]["emb_dim"] for c in cats_info.keys()]  # renamed for clarity

        self.cat_embs = nn.ModuleList([
            nn.Embedding(num_cats, emb_dim)
            for num_cats, emb_dim in zip(cat_cards, cat_emb_dims)
        ])

        late_fusion_width = d_model + sum(cat_emb_dims)
        self.linear_post_fusion = nn.Linear(late_fusion_width, d_model)
        shared_hdim = d_model // 2   # to try: smaller/larger ???
        head_hdim = shared_hdim //2

        layers = []
        in_dim = d_model

        for _ in range(3):
            layers.append(nn.Linear(in_dim, shared_hdim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.1))
            in_dim = shared_hdim

        self.shared_trunk = nn.Sequential(*layers)

        self.heads = nn.ModuleDict({
            "bias": nn.Sequential(
                nn.Linear(shared_hdim, head_hdim),
                nn.ReLU(),
                nn.Linear(head_hdim, 1)
            ),
            "ambi": nn.Sequential(
                nn.Linear(shared_hdim, head_hdim),
                nn.ReLU(),
                nn.Linear(head_hdim, 1),
                nn.Sigmoid()
            )
        })

        self.loss_fct_ambi = nn.MSELoss()
        self.loss_fct_bias = nn.BCEWithLogitsLoss()

    def forward(
        self,
        input_ids,
        attention_mask,
        x_cats,              # [batch, n_cat_vars]
        y_bias=None,         # [batch] float {0,1}
        y_ambi=None          # [batch] float in [0,1]
    ):

        outputs = self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        #last token approach, is effecitvely CLS token
        # h_text = outputs.last_hidden_state[:, -1, :]  # [B, d_model]
        
        #masked pooled approach
        last_hidden = outputs.last_hidden_state # [B, L, d_model]
        mask = attention_mask.unsqueeze(-1)            # [B, L, 1]
        hidden_masked = last_hidden * mask             # [B, L, d_model]
        sum_hidden = hidden_masked.sum(dim=1)          # [B, d_model]
        lengths = mask.sum(dim=1).clamp(min=1)         # [B, 1]
        h_text = sum_hidden / lengths                  # [B, d_model]


        # CATEMBS
        cat_embs = []
        for i, emb_layer in enumerate(self.cat_embs):
            cat_i = x_cats[:, i]           # [B]
            e_i = emb_layer(cat_i)         # [B, emb_dim_i]
            cat_embs.append(e_i)
        e_cat = torch.cat(cat_embs, dim=-1)  # [B, sum(emb_dims)]
        
        h_fused = torch.cat([h_text, e_cat], dim=-1) # [B, late_fusion_width]
        h_proj = self.linear_post_fusion(h_fused)    # [B, d_model]
        h_shared = self.shared_trunk(h_proj)         # [B, shared_hdim]

        logits_bias = self.heads["bias"](h_shared).squeeze(-1)   # [B]
        logits_ambi = self.heads["ambi"](h_shared).squeeze(-1)   # [B]

        loss = None
        if y_bias is not None and y_ambi is not None:
            loss_bias = self.loss_fct_bias(logits_bias, y_bias)
            loss_ambi = self.ambi_loss_weight * self.loss_fct_ambi(logits_ambi, y_ambi)
            loss = loss_bias + loss_ambi

        return {
            "loss": loss,
            "logits_bias": logits_bias,
            "logits_ambi": logits_ambi,
        }
        

In [5]:

SUPPORTED_MODELS = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "meta-llama/Llama-2-7b-hf",
    "Qwen/Qwen-7B",
    "microsoft/deberta-v3-large",
    "xlm-roberta-base",
    "mediabiasgroup/magpie-pt",
]

SUPPORTED_AMBI_TYPES = ["H_obs", "H_cond", "H_model", "H_ambi"]

REGISTRY_PATH = "./model_registry.json"

def load_registry():
    if not Path(REGISTRY_PATH).exists():
        return []
    with open(REGISTRY_PATH) as f:
        return json.load(f)

def save_to_registry(args, run_name, output_dir, metrics):
    registry = load_registry()
    entry = {
        "run_name": run_name,
        "timestamp": datetime.now().isoformat(),
        "output_dir": output_dir,
        # config
        "model_name": args.model_name,
        "ambi_type": args.ambi_type,
        "epochs": args.epochs,
        "lr": args.lr,
        "use_lora": args.use_lora,
        "lora_r": args.lora_r if args.use_lora else None,
        "zero_cats": args.zero_cats,
        "ambi_loss_weight": args.ambi_loss_weight,
        "seed": args.seed,
        # artifacts
        "weights_path": os.path.join(output_dir, "pytorch_model.bin"),
        "preds_path": os.path.join(output_dir, "test_preds.csv"),
        "metrics_path": os.path.join(output_dir, "metrics.json"),
        "confusion_matrix_path": os.path.join(output_dir, "confusion_matrix.png"),
        # results inline for quick lookup
        "metrics": metrics,
    }
    registry.append(entry)
    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)
    print(f"Registered: {run_name}")

def parse_args():
    p = argparse.ArgumentParser(description="Train MTL bias/ambivalence classifier")
    p.add_argument("--model_name",      type=str,   default="meta-llama/Llama-2-7b-hf", choices=SUPPORTED_MODELS)
    p.add_argument("--ambi_type",       type=str,   default="H_model",                  choices=SUPPORTED_AMBI_TYPES)
    p.add_argument("--epochs",          type=int,   default=3)
    p.add_argument("--lr",              type=float, default=5e-5)
    p.add_argument("--batch_size",      type=int,   default=16)
    p.add_argument("--ambi_loss_weight",type=float, default=1.0,  help="Weight on ambivalence loss term")
    p.add_argument("--max_length",      type=int,   default=128)
    p.add_argument("--use_lora",        action="store_true")
    p.add_argument("--lora_r",          type=int,   default=16)
    p.add_argument("--test_size",       type=float, default=0.2)
    p.add_argument("--seed",            type=int,   default=42)
    p.add_argument("--wandb_project",   type=str,   default="bias-ambivalence-mtl")
    p.add_argument("--no_wandb",        action="store_true")
    p.add_argument("--output_dir",      type=str,   default="./model")
    p.add_argument("--test_mode",     type=lambda x: x.lower() == 'true', default=False)
    p.add_argument("--zero_cats", action="store_true", default=False)
    return p.parse_args()


def get_run_name(args):
    model_short = args.model_name.split("/")[-1]
    lora_tag = f"_lora{args.lora_r}" if args.use_lora else ""
    return f"{model_short}_{args.ambi_type}{lora_tag}_{datetime.now().strftime('%m%d_%H%M')}"


def build_compute_metrics():
    acc_metric = evaluate.load("accuracy")
    f1_metric  = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits_bias, logits_ambi = eval_pred.predictions
        labels_bias, labels_ambi = eval_pred.label_ids

        logits_bias  = np.asarray(logits_bias).reshape(-1)
        logits_ambi  = np.asarray(logits_ambi).reshape(-1)
        labels_bias  = np.asarray(labels_bias).reshape(-1)
        labels_ambi  = np.asarray(labels_ambi).reshape(-1)

        probs_bias = 1 / (1 + np.exp(-logits_bias))
        preds_bias = (probs_bias > 0.5).astype(int)

        mse  = np.mean((logits_ambi - labels_ambi) ** 2)
        rmse = np.sqrt(mse)

        return {
            "bias_acc":  acc_metric.compute(predictions=preds_bias, references=labels_bias)["accuracy"],
            "bias_f1":   f1_metric.compute( predictions=preds_bias, references=labels_bias)["f1"],
            "ambi_rmse": rmse,
        }

    return compute_metrics


def train(args, fdf, cat_embs_info):
    run_name   = get_run_name(args)
    output_dir = os.path.join(args.output_dir, run_name)
    os.makedirs(output_dir, exist_ok=True)

    # --- wandb ---
    report_to = ["none"]
    if not args.no_wandb:
        wandb.init(
            project=args.wandb_project,
            name=run_name,
            config=vars(args),
        )
        report_to = ["wandb"]

    # --- tokenizer & model ---
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = MTLClassifier(
        cats_info=cat_embs_info,
        llm_pretrained_name=args.model_name,
        use_lora=args.use_lora,
        lora_r=args.lora_r,
        ambi_loss_weight=args.ambi_loss_weight,
    )

    # --- data ---
    if args.test_mode:
        ffdf = fdf.sample(1000)
    else:
        ffdf = fdf.copy()
    train_df, test_df = train_test_split(
        ffdf, 
        test_size=args.test_size, 
        random_state=args.seed, 
        shuffle=True
    )
    train_dataset = MTLDataset(train_df, cat_embs_info, args.ambi_type, tokenizer, 
                               max_length=args.max_length, zero_cats=args.zero_cats)
    eval_dataset  = MTLDataset(test_df,  cat_embs_info, args.ambi_type, tokenizer, 
                               max_length=args.max_length, zero_cats=args.zero_cats)

    # --- training args ---
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size * 2,
        num_train_epochs=args.epochs,
        weight_decay=0.01,
        remove_unused_columns=False,
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=1,
        report_to=report_to,
        label_names=["y_bias", "y_ambi"],
        seed=args.seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=build_compute_metrics()
    )

    trainer.train()
    trainer.save_model(output_dir)

    


        # cleanup
    test_preds = trainer.predict(eval_dataset)
    logits_bias, logits_ambi = test_preds.predictions
    probs = 1 / (1 + np.exp(-np.array(logits_bias).reshape(-1)))
    preds = (probs > 0.5).astype(int)
    labels = np.array(test_preds.label_ids[0]).reshape(-1)
    
    pred_df = test_df[["text", "label_binary"]].copy()
    pred_df["pred_bias"] = preds
    pred_df["pred_ambi"] = np.array(logits_ambi).reshape(-1)
    pred_df.to_csv(os.path.join(output_dir, "test_preds.csv"), index=False)
    
    # save confusion matrix
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
    import matplotlib.pyplot as plt
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Non-biased", "Biased"])
    disp.plot()
    plt.savefig(os.path.join(output_dir, "confusion_matrix.png"))
    plt.close()
    
    # save metrics
    final_metrics = test_preds.metrics
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump(final_metrics, f, indent=2)
    
    save_to_registry(args, run_name, output_dir, final_metrics)
    if not args.no_wandb:
        wandb.finish()

    return trainer




In [11]:

TEST_MODE = "false"
# 1. baseline — no ambi, no cats
# sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--ambi_loss_weight", "0.0", "--zero_cats", "--test_mode", TEST_MODE]
# args = parse_args()
# train(args, fdf, cat_embs_info)

# # 2. cats only
# sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--ambi_loss_weight", "0.0", "--test_mode", TEST_MODE]
# args = parse_args()
# train(args, fdf, cat_embs_info)

# # 3. ambi only
# sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--zero_cats", "--test_mode", TEST_MODE]
# args = parse_args()
# train(args, fdf, cat_embs_info)

# # 4. full
sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)



wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /ihome/xli/sek188/.netrc.
wandb: Currently logged in as: sek188 (teamMaverick) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/scratch/slurm-1657644/ipykernel_2907125/3247114206.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.693000,0.690410,0.592968,0.744482,0.123966
2,0.673400,0.688188,0.592968,0.744482,0.124594
3,0.634600,0.647539,0.653727,0.718371,0.116521


Registered: TinyLlama-1.1B-Chat-v1.0_H_model_0409_1920


eval/ambi_rmse,▇█▁
eval/bias_acc,▁▁█
eval/bias_f1,██▁
eval/loss,██▁
eval/runtime,█▁▆
eval/samples_per_second,▁█▃
eval/steps_per_second,▁█▃
test/ambi_rmse,▁
test/bias_acc,▁
test/bias_f1,▁
+9,...


In [7]:
TEST_MODE = "false"
# 3. ambi only
sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--zero_cats", "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /ihome/xli/sek188/.netrc.
wandb: Currently logged in as: sek188 (teamMaverick) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/scratch/slurm-1658252/ipykernel_2071927/3247114206.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.670200,0.657179,0.607032,0.606368,0.114661
2,0.617300,0.633267,0.668636,0.738106,0.114450
3,0.582000,0.639281,0.671167,0.744816,0.115057


Registered: TinyLlama-1.1B-Chat-v1.0_H_model_0410_0704


eval/ambi_rmse,▃▁█
eval/bias_acc,▁██
eval/bias_f1,▁██
eval/loss,█▁▃
eval/runtime,▇▁█
eval/samples_per_second,▂█▁
eval/steps_per_second,▂█▁
test/ambi_rmse,▁
test/bias_acc,▁
test/bias_f1,▁
+9,...


In [8]:
# 1. baseline — no ambi, no cats
sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--ambi_loss_weight", "0.0", "--zero_cats", "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)



/scratch/slurm-1658252/ipykernel_2071927/3247114206.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.680400,0.668388,0.592968,0.744482,0.398701
2,0.613400,0.626531,0.641632,0.681500,0.401619
3,0.562900,0.614915,0.670323,0.733878,0.411191


Registered: TinyLlama-1.1B-Chat-v1.0_H_model_0410_0746


eval/ambi_rmse,▁▃█
eval/bias_acc,▁▅█
eval/bias_f1,█▁▇
eval/loss,█▃▁
eval/runtime,▄█▁
eval/samples_per_second,▅▁█
eval/steps_per_second,▅▁█
test/ambi_rmse,▁
test/bias_acc,▁
test/bias_f1,▁
+9,...


In [8]:
TEST_MODE = "false"
# # 2. cats only
sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--ambi_loss_weight", "0.0", "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)
import gc
torch.cuda.empty_cache()
gc.collect()

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /ihome/xli/sek188/.netrc.
wandb: Currently logged in as: sek188 (teamMaverick) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/scratch/slurm-1658252/ipykernel_2779958/3247114206.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.680700,0.676768,0.592968,0.744482,0.360174
2,0.637100,0.657876,0.604219,0.749064,0.349755
3,0.587500,0.629980,0.649508,0.716044,0.341192


Registered: TinyLlama-1.1B-Chat-v1.0_H_model_0410_0946


eval/ambi_rmse,█▄▁
eval/bias_acc,▁▂█
eval/bias_f1,▇█▁
eval/loss,█▅▁
eval/runtime,▁▇█
eval/samples_per_second,█▂▁
eval/steps_per_second,█▂▁
test/ambi_rmse,▁
test/bias_acc,▁
test/bias_f1,▁
+9,...


NameError: name 'trainer' is not defined

In [9]:
# TEST_MODE = "false"
# # # 5. llama 7b
# sys.argv = ["train.py", "--model_name", "meta-llama/Llama-2-7b-hf", "--test_mode", TEST_MODE]
# args = parse_args()
# train(args, fdf, cat_embs_info)
import gc


#note on apr 14 noon - skipping this entirely. i dont rly think it's necessary, we really just need deberta and one large model, and qwen is fine, or the fine tuned is probalby even better. 
torch.cuda.empty_cache()
gc.collect()


10683

In [14]:
# # 6. llama 7b
sys.argv = ["train.py", "--model_name", "Qwen/Qwen-7B", "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)
import gc
torch.cuda.empty_cache()
gc.collect()

#also skipping too. if we have time get multiple cards so no ooom

The repository Qwen/Qwen-7B contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Qwen/Qwen-7B .
 You can inspect the repository content at https://hf.co/Qwen/Qwen-7B.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


qwen.tiktoken: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/911 [00:00<?, ?B/s]

The repository Qwen/Qwen-7B contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Qwen/Qwen-7B .
 You can inspect the repository content at https://hf.co/Qwen/Qwen-7B.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


configuration_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen-7B:
- configuration_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ValueError: Unrecognized configuration class <class 'transformers_modules.Qwen.Qwen_hyphen_7B.ef3c5c9c57b252f3149c1408daf4d649ec8b6c85.configuration_qwen.QWenConfig'> for this kind of AutoModel: AutoModel.
Model type should be one of Aimv2Config, Aimv2VisionConfig, AlbertConfig, AlignConfig, AltCLIPConfig, ApertusConfig, ArceeConfig, AriaConfig, AriaTextConfig, ASTConfig, AutoformerConfig, AyaVisionConfig, BambaConfig, BarkConfig, BartConfig, BeitConfig, BertConfig, BertGenerationConfig, BigBirdConfig, BigBirdPegasusConfig, BioGptConfig, BitConfig, BitNetConfig, BlenderbotConfig, BlenderbotSmallConfig, BlipConfig, Blip2Config, Blip2QFormerConfig, BloomConfig, BltConfig, BridgeTowerConfig, BrosConfig, CamembertConfig, CanineConfig, ChameleonConfig, ChineseCLIPConfig, ChineseCLIPVisionConfig, ClapConfig, CLIPConfig, CLIPTextConfig, CLIPVisionConfig, CLIPSegConfig, ClvpConfig, LlamaConfig, CodeGenConfig, CohereConfig, Cohere2Config, Cohere2VisionConfig, ConditionalDetrConfig, ConvBertConf

In [15]:
# !{sys.executable} -m pip install sentencepiece
!{sys.executable} -m pip uninstall tiktoken -y

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: tiktoken 0.12.0
Uninstalling tiktoken-0.12.0:
  Successfully uninstalled tiktoken-0.12.0
ERROR: Exception:
Traceback (most recent call last):
  File "/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/pip/_internal/cli/base_command.py", line 180, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/pip/_internal/commands/uninstall.py", line 110, in run
    uninstall_pathset.commit()
  File "/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/pip/_internal/req/req_uninstall.py", line 432, in commit
    self._moved_paths.commit()
  File "/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/pip/_internal/req/req_uninstall.py", line 278, in commit
    save_dir.cleanup()
  File "/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/pip/_internal/utils/temp_dir.py", line 173, in cleanup
    rmtree(self._p

In [6]:
TEST_MODE = "false"
# # 7. deberta
sys.argv = ["train.py", "--model_name", "microsoft/deberta-v3-large", "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)
import gc
torch.cuda.empty_cache()
gc.collect()



wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /ihome/xli/sek188/.netrc.
wandb: Currently logged in as: sek188 (teamMaverick) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/scratch/slurm-1664383/ipykernel_2624756/1376205500.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.691400,0.687955,0.592968,0.744482,0.114993
2,0.671700,0.685290,0.592968,0.744482,0.116786
3,0.678400,0.680404,0.593530,0.744655,0.114429


Registered: deberta-v3-large_H_model_0414_1233


eval/ambi_rmse,▃█▁
eval/bias_acc,▁▁█
eval/bias_f1,▁▁█
eval/loss,█▆▁
eval/runtime,▁█▇
eval/samples_per_second,█▁▂
eval/steps_per_second,█▁▂
test/ambi_rmse,▁
test/bias_acc,▁
test/bias_f1,▁
+9,...


3247

In [7]:
# # 8. roberta
sys.argv = ["train.py", "--model_name", "xlm-roberta-base", "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)
import gc
torch.cuda.empty_cache()
gc.collect()


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

/scratch/slurm-1664383/ipykernel_2624756/1376205500.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.691000,0.688351,0.592968,0.744482,0.122101
2,0.665200,0.680204,0.597750,0.742898,0.115336
3,0.665800,0.678436,0.599719,0.734267,0.114410


Registered: xlm-roberta-base_H_model_0414_1254


eval/ambi_rmse,█▂▁
eval/bias_acc,▁▆█
eval/bias_f1,█▇▁
eval/loss,█▂▁
eval/runtime,▁▆█
eval/samples_per_second,█▃▁
eval/steps_per_second,█▃▁
test/ambi_rmse,▁
test/bias_acc,▁
test/bias_f1,▁
+9,...


OSError: [Errno 122] Disk quota exceeded

In [13]:

# # 9. finetuned deberta
sys.argv = ["train.py", "--model_name", "mediabiasgroup/magpie-pt", "--test_mode", TEST_MODE, "--lr", "1e-5", "--epochs", "10"]
args = parse_args()
train(args, fdf, cat_embs_info)
import gc

torch.cuda.empty_cache()
gc.collect()

Some weights of RobertaModel were not initialized from the model checkpoint at mediabiasgroup/magpie-pt and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/scratch/slurm-1664383/ipykernel_2624756/1376205500.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.633300,0.606426,0.684951,0.735350,0.113651
2,0.577900,0.604079,0.688326,0.736065,0.108570
3,0.556900,0.614905,0.683263,0.737039,0.104817
4,0.582300,0.632404,0.678481,0.745151,0.108026
5,0.579900,0.627232,0.680450,0.749559,0.097763
6,0.577700,0.642764,0.682138,0.742362,0.098476
7,0.532700,0.636051,0.681294,0.737913,0.093342
8,0.531400,0.663484,0.682419,0.745434,0.094582
9,0.531000,0.657591,0.680450,0.739330,0.091294
10,0.509800,0.665971,0.679044,0.738483,0.091312


Registered: magpie-pt_H_model_0414_1313


eval/ambi_rmse,█▆▅▆▃▃▂▂▁▁
eval/bias_acc,▆█▄▁▂▄▃▄▂▁
eval/bias_f1,▁▁▂▆█▄▂▆▃▃
eval/loss,▁▁▂▄▄▅▅█▇█
eval/runtime,▁▄▆▇█▇█▇██
eval/samples_per_second,█▅▃▂▁▂▁▂▁▁
eval/steps_per_second,█▅▃▂▁▂▁▂▁▁
test/ambi_rmse,▁
test/bias_acc,▁
test/bias_f1,▁
+9,...


13206